# `mutate` — Reference

`mutate` creates or overwrites columns from a `qry()`-style spec string, each entry evaluated in order via pandas `eval()` — a plain formula per column, no lambda required.

| Syntax | Meaning |
|---|---|
| `"new_col: expr"` | one derived column |
| `"a: expr1, b: expr2"` | several in one call (comma-separated) |
| `"'new_col': expr"` | quoting the key is optional, same as `qry()` |
| `"new_col: if_else(cond, true_val, false_val)"` | dplyr-style two-branch conditional |
| `"new_col: case_when(cond1: v1, cond2: v2, default)"` | multi-branch conditional; last bare value is the catch-all |

Column names *inside* the expression must stay unquoted — see the example below.

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — a plain arithmetic formula, no lambda needed
(pt.select(pt.mutate(penguins, 'bmi: body_mass_g / bill_length_mm ** 2'), 'species', 'body_mass_g', 'bill_length_mm', 'bmi').sample(10))

,species,body_mass_g,bill_length_mm,bmi
10,Adelie,3300.0,37.8,2.309566
43,Adelie,4400.0,44.1,2.262432
209,Chinstrap,4050.0,49.3,1.666331
48,Adelie,3450.0,36.0,2.662037
50,Adelie,3500.0,39.6,2.231915
205,Chinstrap,4050.0,50.7,1.575575
191,Chinstrap,4500.0,53.5,1.572190
46,Adelie,3425.0,41.1,2.027575
335,Gentoo,5850.0,55.1,1.926871
326,Gentoo,4700.0,41.7,2.702874


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it
# into a string literal, not a column reference, and breaks the arithmetic
try:
    pt.mutate(penguins, "bmi: 'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as e:
    print('TypeError:', e)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call
(pt.select(pt.mutate(penguins, 'heavy: body_mass_g > 4000, mass_kg: body_mass_g / 1000'), 'species', 'body_mass_g', 'mass_kg', 'heavy').sample(10))

,species,body_mass_g,mass_kg,heavy
329,Gentoo,5500.0,5.500,True
116,Adelie,2900.0,2.900,False
332,Gentoo,4650.0,4.650,True
1,Adelie,3800.0,3.800,False
185,Chinstrap,4100.0,4.100,True
117,Adelie,3775.0,3.775,False
172,Chinstrap,3600.0,3.600,False
195,Chinstrap,3500.0,3.500,False
330,Gentoo,5000.0,5.000,True
147,Adelie,3475.0,3.475,False


In [5]:
# mass_lb references mass_kg, derived by the entry just before it
(pt.select(pt.mutate(penguins, 'mass_kg: body_mass_g / 1000, mass_lb: mass_kg * 2.20462'), 'species', 'mass_kg', 'mass_lb').sample(10))

,species,mass_kg,mass_lb
122,Adelie,3.450,7.605939
257,Gentoo,5.250,11.574255
176,Chinstrap,3.300,7.275246
253,Gentoo,6.050,13.337951
300,Gentoo,4.625,10.196367
142,Adelie,3.050,6.724091
216,Chinstrap,3.400,7.495708
169,Chinstrap,3.700,8.157094
160,Chinstrap,4.150,9.149173
307,Gentoo,5.300,11.684486


## String comparisons, optional key quoting, and local variables
Quoting the key (`'is_adelie'` vs `is_adelie`) is optional, same as `qry()`. String literals *inside* the expression (e.g. `'Adelie'`) still need real quotes — only the column names must stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.

In [6]:
unquoted = pt.mutate(penguins, "is_adelie: species == 'Adelie'")
quoted = pt.mutate(penguins, "'is_adelie': species == 'Adelie'")
(unquoted['is_adelie'] == quoted['is_adelie']).all()

np.True_

In [ ]:
# @-prefixed names resolve against the scope that called mutate(), not mutate()'s own internals
threshold = 4000
(pt.select(pt.mutate(penguins, 'heavy: body_mass_g >= @threshold'), 'species', 'body_mass_g', 'heavy').sample(10))

## Overwriting an existing column

In [7]:
# mutate() can overwrite a column in place, e.g. converting units
(pt.select(pt.mutate(penguins, 'body_mass_g: body_mass_g / 1000'), 'species', 'body_mass_g').sample(10))

,species,body_mass_g
97,Adelie,4.35
95,Adelie,4.30
10,Adelie,3.30
182,Chinstrap,3.20
315,Gentoo,5.20
149,Adelie,3.75
294,Gentoo,4.70
33,Adelie,3.90
262,Gentoo,4.30
274,Gentoo,4.90


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [8]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3000.0, 4000.0], 'bill length mm': [30.0, 40.0]})
pt.mutate(spaced, 'bmi: `body mass g` / `bill length mm` ** 2')

,body mass g,bill length mm,bmi
0,3000.0,30.0,3.333333
1,4000.0,40.0,2.500000


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [9]:
(pt.select(pt.qry(pt.mutate(penguins, 'bmi: body_mass_g / bill_length_mm ** 2'), {'bmi': ('>', 2)}), 'species', 'bmi').sample(10))

,species,bmi
337,Gentoo,2.519484
89,Adelie,2.379049
17,Adelie,2.491349
340,Gentoo,2.214369
108,Adelie,2.187227
11,Adelie,2.589513
306,Gentoo,2.442184
258,Gentoo,2.246901
336,Gentoo,2.461810
107,Adelie,2.672624


## Conditional column creation — `if_else()` and `case_when()`
Plain `eval()` has no ternary/`where()` support, but `mutate()` recognizes two dplyr-style expression forms by name and evaluates them via `np.where()`/`np.select()` instead: `if_else(condition, true_value, false_value)` and `case_when(cond1: val1, cond2: val2, ..., default)`. A last argument with no colon is the catch-all default (like SQL ELSE). Conditions/non-string values are still `eval()` expressions; string outcomes need quotes.

In [10]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
(pt.select(pt.mutate(penguins, "weight_class: if_else(body_mass_g > 4000, 'heavy', 'light')"), 'species', 'body_mass_g', 'weight_class').sample(10))

,species,body_mass_g,weight_class
232,Gentoo,4650.0,heavy
335,Gentoo,5850.0,heavy
194,Chinstrap,3550.0,light
84,Adelie,3350.0,light
174,Chinstrap,2900.0,light
213,Chinstrap,3650.0,light
51,Adelie,4300.0,heavy
134,Adelie,3425.0,light
1,Adelie,3800.0,light
178,Chinstrap,3400.0,light


In [11]:
# case_when(cond1: val1, cond2: val2, ..., default) — last bare value is the catch-all
# checked in order, first match wins; `True` is an optional catch-all default and must be listed last
(pt.select(pt.mutate(penguins, "size_class: case_when(body_mass_g >= 4500: 'large', body_mass_g >= 3500: 'medium', 'small')"), 'species', 'body_mass_g', 'size_class').sample(10))

,species,body_mass_g,size_class
340,Gentoo,4850.0,large
267,Gentoo,5400.0,large
198,Chinstrap,3400.0,small
74,Adelie,3700.0,medium
72,Adelie,3550.0,medium
29,Adelie,3950.0,medium
210,Chinstrap,3800.0,medium
237,Gentoo,6300.0,large
343,Gentoo,5400.0,large
332,Gentoo,4650.0,large
